In [ ]:
!pip -q install lightgbm==4.6.0
import os, json, time, gc, warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.metrics import average_precision_score
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass
print('pandas', pd.__version__, '| numpy', np.__version__, '| lightgbm', lgb.__version__)

In [ ]:
SMOKE = True

DATA_DIR = '/content/drive/MyDrive'
OUT_DIR  = '/content/drive/MyDrive/THAT_IDS_RERUN/journal_step21_lgbm_group_disjoint_v1'

DATASETS = {
    'UNSW_V2': dict(file='NF-UNSW-NB15-v2.csv', rows=2_390_275, groups=236_999),
    'CIC_V2':  dict(file='NF-CSE-CIC-IDS2018-v2.csv', rows=18_893_708, groups=8_981_062),
}
SPLITS = {
    ('UNSW_V2', 42):  dict(salt=11652151511078254891,
        train=(1670959, 1604425, 66534), validation=(248331, 235791, 12540), test=(470985, 455006, 15979)),
    ('UNSW_V2', 123): dict(salt=4518799303034446728,
        train=(1669960, 1600374, 69586), validation=(219285, 210798, 8487), test=(501030, 484050, 16980)),
    ('UNSW_V2', 456): dict(salt=5346391484498864780,
        train=(1667966, 1600280, 67686), validation=(269673, 260319, 9354), test=(452636, 434623, 18013)),
    ('CIC_V2', 42):   dict(salt=8095878257575067585,
        train=(13232623, 11630102, 1602521), validation=(2012758, 1796541, 216217), test=(3648327, 3208924, 439403)),
    ('CIC_V2', 123):  dict(salt=9558457822766297306,
        train=(13254901, 11705849, 1549052), validation=(1893721, 1659711, 234010), test=(3745086, 3270007, 475079)),
    ('CIC_V2', 456):  dict(salt=3253095538486194220,
        train=(13336590, 11700738, 1635852), validation=(1836131, 1621337, 214794), test=(3720987, 3313492, 407495)),
}
EXCLUDE_COLS = ['FLOW_START_MILLISECONDS', 'FLOW_END_MILLISECONDS', 'IPV4_SRC_ADDR',
                'IPV4_DST_ADDR', 'L4_SRC_PORT', 'L4_DST_PORT', 'Label', 'Attack']
N_FEATURES = 39
CHUNK = 1_000_000

LGBM_PARAMS = dict(objective='binary', n_estimators=2000, learning_rate=0.03, num_leaves=31,
                   max_depth=-1, min_child_samples=50, subsample=0.8, subsample_freq=1,
                   colsample_bytree=0.8, reg_alpha=0.0, reg_lambda=1.0, max_bin=255,
                   deterministic=True, force_col_wise=True, n_jobs=-1, verbose=-1)
EARLY_STOP = 100
# families shown in Table 11 (dataset, name as spelled in the Attack column, label in the paper)
TABLE11 = [('UNSW_V2', 'Exploits', 'Exploits (UNSW)'), ('UNSW_V2', 'Analysis', 'Analysis (UNSW)'),
           ('CIC_V2', 'Bot', 'Bot (CIC)'), ('CIC_V2', 'Infilteration', 'Infiltration (CIC)')]

if SMOKE:
    RUN = [('UNSW_V2', 42)]
    LGBM_PARAMS['n_estimators'] = 300
    OUT_DIR = OUT_DIR + '_smoke'
else:
    RUN = list(SPLITS)
os.makedirs(OUT_DIR, exist_ok=True)
print('runs:', RUN); print('output:', OUT_DIR)

In [ ]:
def load_dataset(name):
    cfg = DATASETS[name]
    path = os.path.join(DATA_DIR, cfg['file'])
    if not os.path.exists(path):
        raise FileNotFoundError(f'{path} not found: edit DATA_DIR in the config cell')
    header = pd.read_csv(path, nrows=0).columns.tolist()
    feat_cols = [c for c in header if c not in EXCLUDE_COLS]
    assert len(feat_cols) == N_FEATURES, f'{len(feat_cols)} feature columns, expected {N_FEATURES}'
    n = cfg['rows']
    X = np.empty((n, N_FEATURES), dtype=np.float32)
    keys = np.empty(n, dtype=np.uint64)
    y = np.empty(n, dtype=np.int8)
    fam = np.empty(n, dtype=np.int16)
    fam_names, pos, t0 = {}, 0, time.time()
    for chunk in pd.read_csv(path, usecols=feat_cols + ['Label', 'Attack'], chunksize=CHUNK):
        f = chunk[feat_cols].apply(pd.to_numeric, errors='coerce').astype(np.float32)
        a = f.to_numpy(copy=True)
        a[~np.isfinite(a)] = 0.0
        f = pd.DataFrame(a, columns=feat_cols)
        m = len(f)
        if pos + m > n:
            raise ValueError(f'{name}: more rows than expected ({n})')
        X[pos:pos + m] = a
        keys[pos:pos + m] = pd.util.hash_pandas_object(f, index=False).to_numpy()
        y[pos:pos + m] = chunk['Label'].to_numpy().astype(np.int8)
        att = chunk['Attack'].astype(str)
        for v in att.unique():
            fam_names.setdefault(v, len(fam_names))
        fam[pos:pos + m] = att.map(fam_names).to_numpy().astype(np.int16)
        pos += m
        print(f'\r{name}: {pos:,} rows read ({time.time() - t0:.0f}s)', end='')
    print()
    assert pos == n, f'{name}: read {pos:,} rows, expected {n:,}'
    n_groups = len(np.unique(keys))
    print(f'{name}: groups {n_groups:,} (expected {cfg["groups"]:,}) | '
          f'repeated-row rate {100 * (1 - n_groups / n):.2f}% | attack {100 * y.mean():.2f}%')
    if n_groups != cfg['groups']:
        print('  WARNING: group count differs from the audit, so preprocessing does not match the paper.')
    names = [k for k, _ in sorted(fam_names.items(), key=lambda kv: kv[1])]
    return X, keys, y, fam, names, feat_cols, n_groups

In [ ]:
U64 = np.uint64
def _mix(z, gamma):
    z = z.copy()
    with np.errstate(over='ignore'):
        if gamma:
            z += U64(0x9E3779B97F4A7C15)
        z = (z ^ (z >> U64(30))) * U64(0xBF58476D1CE4E5B9)
        z = (z ^ (z >> U64(27))) * U64(0x94D049BB133111EB)
    return z ^ (z >> U64(31))

VARIANTS = [(comb, gamma, mapping) for comb in ('xor', 'add') for gamma in (True, False)
            for mapping in ('div64', 'top53')]

def assign(keys, salt, variant):
    comb, gamma, mapping = variant
    with np.errstate(over='ignore'):
        x = keys ^ U64(salt) if comb == 'xor' else keys + U64(salt)
    z = _mix(x, gamma)
    u = z.astype(np.float64) / 2.0**64 if mapping == 'div64' else (z >> U64(11)).astype(np.float64) / 2.0**53
    return np.where(u < 0.70, 0, np.where(u < 0.80, 1, 2)).astype(np.int8)

def counts(part, y):
    return {n: (int((part == i).sum()), int(((part == i) & (y == 0)).sum()), int(((part == i) & (y == 1)).sum()))
            for i, n in enumerate(('train', 'validation', 'test'))}

def rebuild_split(name, seed, keys, y):
    ref = SPLITS[(name, seed)]
    want = {n: tuple(ref[n]) for n in ('train', 'validation', 'test')}
    tried = []
    for v in VARIANTS:
        part = assign(keys, ref['salt'], v)
        got = counts(part, y)
        if got == want:
            print(f'[{name} seed {seed}] split reproduced exactly with variant {v}: '
                  f'train {want["train"][0]:,} | val {want["validation"][0]:,} | test {want["test"][0]:,}')
            return part, v
        tried.append((v, got['train'][0], got['validation'][0], got['test'][0]))
    msg = '\n'.join(f'  {v}: train {a:,} val {b:,} test {c:,}' for v, a, b, c in tried)
    raise RuntimeError(f'{name} seed {seed}: no variant reproduces split_metadata.json '
                       f'(want train {want["train"][0]:,} val {want["validation"][0]:,} test {want["test"][0]:,}).\n'
                       f'{msg}\nDo not train on a different partition. Send the split function from the '
                       f'step4 notebook so the mixing step can be matched.')

In [ ]:
def best_f1_macro_threshold(y, s):
    # exact search over distinct validation scores; prediction is score >= threshold
    order = np.argsort(-s, kind='stable')
    ys, ss = y[order].astype(np.int64), s[order]
    tp, fp = np.cumsum(ys), np.cumsum(1 - ys)
    last = np.r_[ss[1:] != ss[:-1], True]
    tp, fp, thr = tp[last], fp[last], ss[last]
    P = ys.sum(); N = len(ys) - P
    fn, tn = P - tp, N - fp
    f1 = 0.5 * (2 * tp / (2 * tp + fp + fn) + 2 * tn / (2 * tn + fn + fp))
    return float(thr[np.argmax(f1)])

def metrics(y, s, pred):
    y = y.astype(np.int64); pred = pred.astype(np.int64)
    tp = int((pred & y).sum()); fp = int((pred & (1 - y)).sum())
    fn = int(((1 - pred) & y).sum()); tn = int(((1 - pred) & (1 - y)).sum())
    div = lambda a, b: a / b if b else float('nan')
    out = dict(n=len(y), n_attack=tp + fn, n_benign=fp + tn,
               attack_pct=100 * div(tp + fn, len(y)),
               f1_macro=0.5 * (div(2 * tp, 2 * tp + fp + fn) + div(2 * tn, 2 * tn + fn + fp)),
               fpr_pct=100 * div(fp, fp + tn), tpr=div(tp, tp + fn), precision=div(tp, tp + fp),
               tp=tp, fp=fp, fn=fn, tn=tn)
    out['pr_auc'] = (float(average_precision_score(y, s))
                     if s is not None and 0 < tp + fn < len(y) else float('nan'))
    return out

def family_recall(y, pred, fam, names):
    rows = []
    for f in np.unique(fam[y == 1]):
        m = (fam == f) & (y == 1)
        rows.append(dict(family=names[f], count=int(m.sum()), recall=float(pred[m].mean())))
    return rows

In [ ]:
def run_one(name, seed, data):
    X, keys, y, fam, names, feat_cols, n_groups = data
    part, variant = rebuild_split(name, seed, keys, y)
    tr, va, te = (np.flatnonzero(part == i) for i in range(3))
    assert not np.isin(keys[te], np.unique(keys[tr])).any(), 'test vector found in train'
    assert not np.isin(keys[va], np.unique(keys[tr])).any(), 'validation vector found in train'

    Xtr = X[tr]; mn, mx = Xtr.min(0), Xtr.max(0)
    rng_ = np.where(mx > mn, mx - mn, 1.0).astype(np.float32)
    Xtr -= mn; Xtr /= rng_
    Xva = (X[va] - mn) / rng_
    Xte = (X[te] - mn) / rng_
    ytr, yva, yte, fte = y[tr], y[va], y[te], fam[te]

    spw = min(float((ytr == 0).sum()) / max(int((ytr == 1).sum()), 1), 12.0)
    model = lgb.LGBMClassifier(**LGBM_PARAMS, scale_pos_weight=spw, random_state=seed,
                               bagging_seed=seed, feature_fraction_seed=seed, data_random_seed=seed)
    t0 = time.time()
    model.fit(Xtr, ytr, eval_set=[(Xva, yva)], eval_metric='binary_logloss',
              callbacks=[lgb.early_stopping(EARLY_STOP, verbose=False)])
    wall = time.time() - t0
    s_va = model.predict_proba(Xva)[:, 1]
    s_te = model.predict_proba(Xte)[:, 1]
    thr = best_f1_macro_threshold(yva, s_va)
    pred = s_te >= thr

    run = dict(dataset=name, seed=seed, split='hash_group_disjoint', model='lightgbm',
               salt=str(SPLITS[(name, seed)]['salt']), mix_variant='/'.join(map(str, variant)),
               threshold=thr, best_iteration=int(model.best_iteration_ or LGBM_PARAMS['n_estimators']),
               scale_pos_weight=spw, train_seconds=round(wall, 1), smoke=SMOKE, **metrics(yte, s_te, pred))
    fams = [dict(dataset=name, seed=seed, **r) for r in family_recall(yte, pred, fte, names)]

    np.savez_compressed(os.path.join(OUT_DIR, f'{name}_seed{seed}_test_predictions.npz'),
                        row_index=te.astype(np.int64), score=s_te.astype(np.float32),
                        label=yte, family=fte, key=keys[te], family_names=np.array(names))
    with open(os.path.join(OUT_DIR, f'{name}_seed{seed}_result.json'), 'w') as fh:
        json.dump(dict(run=run, family_recall=fams, lgbm_params=LGBM_PARAMS), fh, indent=1)
    print(f'  F1-macro {run["f1_macro"]:.4f} | PR-AUC {run["pr_auc"]:.4f} | FPR {run["fpr_pct"]:.4f}% | '
          f'TPR {run["tpr"]:.4f} | best iteration {run["best_iteration"]} | train {wall:.0f}s\n')
    del Xtr, Xva, Xte, model; gc.collect()

for name in dict.fromkeys(d for d, _ in RUN):
    todo = [s for d, s in RUN if d == name
            and not os.path.exists(os.path.join(OUT_DIR, f'{name}_seed{s}_result.json'))]
    if not todo:
        print(f'{name}: all seeds already done, skipping'); continue
    data = load_dataset(name)
    for seed in todo:
        run_one(name, seed, data)
    del data; gc.collect()

In [ ]:
runs, fams = [], []
for fn in sorted(os.listdir(OUT_DIR)):
    if fn.endswith('_result.json'):
        r = json.load(open(os.path.join(OUT_DIR, fn)))
        runs.append(r['run']); fams += r['family_recall']
runs, fams = pd.DataFrame(runs), pd.DataFrame(fams)
runs.to_csv(os.path.join(OUT_DIR, 'step21_all_runs.csv'), index=False)
fams.to_csv(os.path.join(OUT_DIR, 'step21_family_recall_runs.csv'), index=False)

cols = ['f1_macro', 'pr_auc', 'fpr_pct', 'tpr']
summary = runs.groupby('dataset')[cols + ['n', 'n_attack', 'best_iteration']].agg(['mean', 'std'])
summary.to_csv(os.path.join(OUT_DIR, 'step21_summary.csv'))
print('seeds per dataset:', runs.groupby('dataset')['seed'].nunique().to_dict())
print(runs[['dataset', 'seed', 'n', 'n_attack', 'best_iteration'] + cols].round(4).to_string(index=False), '\n')

def pm(m, s, d=4):
    return f'{m:.{d}f} $\\pm$ {s:.{d}f}'
print('Table 10 rows (F1-macro & PR-AUC & FPR (%) & TPR):')
for ds in summary.index:
    r = summary.loc[ds]
    d_fpr = 3 if r[('fpr_pct', 'mean')] >= 0.1 else 4
    print(f'  {ds}: LGBM & {pm(r[("f1_macro","mean")], r[("f1_macro","std")])} & '
          f'{pm(r[("pr_auc","mean")], r[("pr_auc","std")])} & '
          f'{pm(r[("fpr_pct","mean")], r[("fpr_pct","std")], d_fpr)} & '
          f'{pm(r[("tpr","mean")], r[("tpr","std")])} \\\\')

fam_tab = fams.groupby(['dataset', 'family'])[['count', 'recall']].agg(['mean', 'std'])
fam_tab.to_csv(os.path.join(OUT_DIR, 'step21_family_recall_summary.csv'))
print('\nTable 11, LightGBM column (support = mean test count):')
for ds, f, label in TABLE11:
    if (ds, f) in fam_tab.index:
        r = fam_tab.loc[(ds, f)]
        print(f'  {label}: support {r[("count","mean")]:,.0f} | {pm(r[("recall","mean")], r[("recall","std")])}')
print('\nAll families:')
print(fam_tab.round(4).to_string())